# Sendo a base apresentada no arquivo abaixo:
- https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?select=olist_customers_dataset.csv
    - Já disponível no banco de dados `vendas_db.db` através do link:
        - https://drive.google.com/file/d/1eONzrbEu5BoijDRj56gjL_2Ik5qxtR6U/view?usp=sharing
<br><br>
- Sua tarefa é ajudar a área de negócios a **montar uma apresentação para a diretoria** para provar a **necessidade de investistimento em uma área de melhoria da experência do cliente ao ter um atraso na entrega**
<br><br>
- Algumas considerações são importantes
    - O **time de logística não considera que o atraso na entrega é um problema relevante** e falou que, em média, as entregas estão sendo feitas 10 dias antes do prazo combinado
    - Não é desejado a previsão de uma entrega atrasada, apenas a **exposição que esse é um problema que pode impactar os clientes**
    - Não queremos uma abordagem de: "nenhuma entrega pode atrasar". Vamos ser mais tranquilos e seguir na linha de: **"uma entrega pode atrasar. Como eu posso melhorar a experiência do cliente caso isso aconteça?"**

In [1]:
import sqlite3 
import pandas as pd

con = sqlite3.connect('../data/vendas_db.db')

cur = con.cursor()

In [2]:
def executa_consulta(consulta):
    resultado = cur.execute(consulta).fetchall()
    resultado = pd.DataFrame(resultado)
    colunas = [i[0] for i in cur.description]
    if resultado.shape[1] > 0:
        resultado.columns = colunas
    print(resultado.shape)
    display(resultado.head(3))
    return resultado

In [3]:
total_pedidos = executa_consulta('SELECT COUNT (*) AS total_pedidos_entregues,\
                 ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date)), 2) AS media_dias_vs_prazo\
                 FROM orders\
                 WHERE order_status = "delivered"\
                 AND order_delivered_customer_date IS NOT NULL\
                 AND order_estimated_delivery_date IS NOT NULL')

(1, 2)


,total_pedidos_entregues,media_dias_vs_prazo
0,96470,-11.18


### PASSO 1 

“A logística está correta quando olha a média: os pedidos entregues chegaram, em média, 11 dias antes do prazo. Porém, experiência do cliente não acontece na média. Ela acontece pedido a pedido. Por isso, o próximo passo é separar entregas no prazo e entregas atrasadas.”



## Se a média é boa, ainda assim existem clientes impactados por atraso?


In [4]:
status_entrega = executa_consulta('SELECT \
                 CASE \
                     WHEN order_delivered_customer_date > order_estimated_delivery_date \
                         THEN "Atrasado" \
                     ELSE "No prazo ou adiantado" \
                 END AS status_entrega, \
                 COUNT(*) AS qtd_pedidos, \
                 ROUND(100.0 * COUNT(*) / ( \
                     SELECT COUNT(*) \
                     FROM orders \
                     WHERE order_status = "delivered" \
                       AND order_delivered_customer_date IS NOT NULL \
                       AND order_estimated_delivery_date IS NOT NULL \
                 ), 2) AS porcentagem_pedidos \
                 FROM orders \
                 WHERE order_status = "delivered" \
                   AND order_delivered_customer_date IS NOT NULL \
                   AND order_estimated_delivery_date IS NOT NULL \
                 GROUP BY status_entrega')

(2, 3)


,status_entrega,qtd_pedidos,porcentagem_pedidos
0,Atrasado,7826,8.11
1,No prazo ou adiantado,88644,91.89


“A operação realmente funciona bem para a maioria dos clientes: quase 92% dos pedidos chegam no prazo ou antes. Mas ainda temos 7.826 clientes impactados por atraso. A proposta aqui não é dizer que a logística falha como um todo, e sim que existe um grupo relevante de clientes que precisa de uma experiência melhor quando o atraso acontece.”

## Agora que sabemos que o atraso existe, precisamos entender se ele afeta a satisfação do cliente.


In [5]:
executa_consulta('SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date\
                 THEN "Atrasado"\
                 ELSE "No prazo ou adiantado"\
                 END AS status_entrega,\
                 COUNT(*) AS qtd_reviews,\
                 ROUND(AVG(r.review_score), 2) AS notas_media,\
                 ROUND(100.0 * AVG(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END), 2) AS porcetagem_notas_ruins\
                 FROM orders o\
                 JOIN order_reviews r\
                 ON o.order_id = r.order_id\
                 WHERE o.order_status = "delivered"\
                 AND o.order_delivered_customer_date IS NOT NULL\
                 AND o.order_estimated_delivery_date IS NOT NULL\
                 GROUP BY status_entrega')

(2, 4)


,status_entrega,qtd_reviews,notas_media,porcetagem_notas_ruins
0,Atrasado,7700,2.57,54.03
1,No prazo ou adiantado,88653,4.29,9.23


,status_entrega,qtd_reviews,notas_media,porcetagem_notas_ruins
0,Atrasado,7700,2.57,54.03
1,No prazo ou adiantado,88653,4.29,9.23


Aqui está o ponto central da análise: a operação vai bem na média, mas quando o cliente recebe depois do prazo, a experiência muda completamente. A nota média cai de 4,29 para 2,57, e mais da metade dos clientes com atraso dão nota 1 ou 2. Isso mostra que o atraso é um momento crítico da jornada.

## Se o atraso piora tanto a experiência, precisamos entender se todos os atrasos têm o mesmo impacto ou se atrasos maiores são ainda mais críticos.


In [6]:
faixa_atraso = executa_consulta('SELECT CASE\
                 WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <= 0\
                    THEN "No prazo ou adiantado"\
                 WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 1 AND 3\
                    THEN "Atraso de 1 a 3 dias"\
                 WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 4 AND 7\
                    THEN "Atraso de 4 a 7 dias"\
                 ELSE "Ataso acima de 7 dias"\
                    END AS faixa_atraso,\
                 COUNT(*) AS qtd_pedidos,\
                 ROUND(AVG(r.review_score), 2) AS nota_media,\
                 ROUND(100.0 * AVG(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END), 2) AS porcentagem_notas_ruins\
                 FROM orders o\
                 JOIN order_reviews r\
                 ON o.order_id = r.order_id\
                 WHERE o.order_status = "delivered"\
                 AND o.order_delivered_customer_date IS NOT NULL\
                 AND o.order_estimated_delivery_date IS NOT NULL\
                 GROUP BY faixa_atraso\
                 ORDER BY CASE faixa_atraso\
                 WHEN "No prazo ou adiantado" THEN 1\
                 WHEN "Atraso de 1 a 3 dias" THEN 2\
                 WHEN "Atraso de 4 a 7 dias" THEN 3\
                 ELSE 4\
                 END;')

(4, 4)


,faixa_atraso,qtd_pedidos,nota_media,porcentagem_notas_ruins
0,No prazo ou adiantado,88653,4.29,9.23
1,Atraso de 1 a 3 dias,1360,3.51,25.51
2,Atraso de 4 a 7 dias,1281,2.17,65.57


O impacto do atraso não é linear. Um atraso pequeno já aumenta o risco de insatisfação, mas atrasos acima de 4 dias parecem entrar em uma zona crítica da experiência do cliente. Nessa faixa, mais da metade dos clientes avalia mal a experiência.

A faixa acima de 7 dias tem nota média um pouco melhor que a faixa de 4 a 7 dias, mas ainda é muito baixa. Isso pode acontecer por diferenças de perfil de pedido, região, expectativa do cliente ou porque alguns clientes já esperavam entregas mais longas. O ponto principal é que ambas as faixas têm nível crítico de insatisfação.

In [7]:
status_reviews = executa_consulta('SELECT CASE \
                 WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date \
                 THEN "Atrasado" \
                 ELSE "No prazo ou adiantado" \
                 END AS status_entrega, \
                 COUNT(*) AS qtd_reviews, \
                 SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) AS qtd_clientes_insatisfeitos, \
                 ROUND(100.0 * SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) / COUNT(*), 2) AS porcentagem_insatisfeitos \
                 FROM orders o \
                 JOIN order_reviews r \
                 ON o.order_id = r.order_id \
                 WHERE o.order_status = "delivered" \
                 AND o.order_delivered_customer_date IS NOT NULL \
                 AND o.order_estimated_delivery_date IS NOT NULL \
                 GROUP BY status_entrega')

(2, 4)


,status_entrega,qtd_reviews,qtd_clientes_insatisfeitos,porcentagem_insatisfeitos
0,Atrasado,7700,4160,54.03
1,No prazo ou adiantado,88653,8186,9.23


“O atraso não é o maior volume da operação, mas é um grupo de alto risco. Ele gera mais de 4 mil clientes insatisfeitos e tem uma taxa de insatisfação quase seis vezes maior do que entregas no prazo.”



In [8]:
estado_media = executa_consulta('SELECT c.customer_state AS estado, \
                 COUNT(*) AS qtd_pedidos, \
                 SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS qtd_atrasados, \
                 ROUND(100.0 * AVG(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END), 2) AS pct_atraso, \
                 ROUND(AVG(r.review_score), 2) AS nota_media \
                 FROM orders o \
                 JOIN customers c \
                 ON o.customer_id = c.customer_id \
                 JOIN order_reviews r \
                 ON o.order_id = r.order_id \
                 WHERE o.order_status = "delivered" \
                 AND o.order_delivered_customer_date IS NOT NULL \
                 AND o.order_estimated_delivery_date IS NOT NULL \
                 GROUP BY c.customer_state \
                 HAVING COUNT(*) >= 500 \
                 ORDER BY pct_atraso DESC')

(17, 5)


,estado,qtd_pedidos,qtd_atrasados,pct_atraso,nota_media
0,MA,716,137,19.13,3.84
1,CE,1276,195,15.28,3.94
2,BA,3246,447,13.77,3.93


In [9]:
categoria_atraso = executa_consulta('SELECT \
                 CASE \
                     WHEN p.product_category_name IS NULL THEN "Categoria não informada" \
                     ELSE p.product_category_name \
                 END AS categoria, \
                 COUNT(*) AS qtd_pedidos, \
                 SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS qtd_atrasados, \
                 ROUND(100.0 * AVG(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END), 2) AS pct_atraso, \
                 ROUND(AVG(r.review_score), 2) AS nota_media \
                 FROM orders o \
                 JOIN order_reviews r \
                 ON o.order_id = r.order_id \
                 JOIN order_items i \
                 ON o.order_id = i.order_id \
                 JOIN products p \
                 ON i.product_id = p.product_id \
                 WHERE o.order_status = "delivered" \
                 AND o.order_delivered_customer_date IS NOT NULL \
                 AND o.order_estimated_delivery_date IS NOT NULL \
                 GROUP BY categoria \
                 HAVING COUNT(*) >= 500 \
                 ORDER BY pct_atraso DESC')

(28, 5)


,categoria,qtd_pedidos,qtd_atrasados,pct_atraso,nota_media
0,eletronicos,2711,264,9.74,4.07
1,Categoria não informada,1533,141,9.20,3.94
2,beleza_saude,9456,838,8.86,4.19
